In [180]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torchinfo

Understanding training pipeline with real world dataset

Getting data

In [181]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [182]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [183]:
df.describe()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,0.0
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,NaN
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,NaN
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,NaN
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,NaN
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,NaN
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,NaN
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,NaN


In [184]:
df.shape

(569, 33)

In [185]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [186]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Splitting data

In [187]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[: , 0], test_size= 0.3)

Scaling data

In [188]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [189]:
X_train

array([[ 0.27818288,  2.35076359,  0.18772099, ..., -0.76111808,
         0.60098497, -1.2181116 ],
       [ 1.64029279,  0.51524711,  1.58261646, ...,  1.01108803,
         0.32276143, -0.0426344 ],
       [-0.72517003,  1.15600923, -0.67230248, ...,  0.2305551 ,
        -0.13297896,  0.92193182],
       ...,
       [-0.47724853, -0.26123198, -0.54334124, ..., -1.5142104 ,
        -1.10590789, -1.58031606],
       [-1.31259816, -1.35809213, -1.19026154, ..., -0.12464836,
        -0.05616878,  2.98604729],
       [ 0.2315153 , -0.51041725,  0.16700262, ..., -0.46994271,
        -0.81402921, -1.0701927 ]])

In [190]:
y_train

414    M
218    M
537    B
565    M
274    M
      ..
206    B
204    B
360    B
505    B
169    B
Name: diagnosis, Length: 398, dtype: object

Label Encoder

In [191]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

NP arrays to Tensors

In [192]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

y_train_tensor

tensor([1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0.,
        1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
        1., 1., 0., 1., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 1., 0., 1., 0.,
        1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0.,
        1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0.,
        0., 1., 0., 1., 0., 0., 1., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0.,
        0., 1., 0., 1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
        1., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 1.,
        0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1.,
        1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1.,
        0., 0., 1., 0., 0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 1., 1., 1., 0.,
        1., 0., 0., 1., 0., 0., 1., 0., 

In [193]:
X_train_tensor

tensor([[ 0.2782,  2.3508,  0.1877,  ..., -0.7611,  0.6010, -1.2181],
        [ 1.6403,  0.5152,  1.5826,  ...,  1.0111,  0.3228, -0.0426],
        [-0.7252,  1.1560, -0.6723,  ...,  0.2306, -0.1330,  0.9219],
        ...,
        [-0.4772, -0.2612, -0.5433,  ..., -1.5142, -1.1059, -1.5803],
        [-1.3126, -1.3581, -1.1903,  ..., -0.1246, -0.0562,  2.9860],
        [ 0.2315, -0.5104,  0.1670,  ..., -0.4699, -0.8140, -1.0702]])

In [194]:
X_train_tensor.shape

torch.Size([398, 30])

In [195]:
X_test_tensor.shape

torch.Size([171, 30])

Data Improving

In [196]:
from torch.utils.data import DataLoader, Dataset

In [197]:
class CustomDataset(Dataset):
    
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
        
        

In [198]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [199]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Defining the model - NN

In [200]:
class NN(nn.Module):
    
    def __init__(self, num_features): # X is input data
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(num_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
        
        
    def forward(self, X):
        out = self.network(X)
        return out


Important params

In [201]:
lr = 0.1
epochs = 50

Training Pipeline

In [202]:
# 1. Create model
model = NN(X_train_tensor.shape[1])
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=lr)
loss = nn.BCELoss()

## These steps in loop - epochs ##

for epoch in range(epochs):    
    
    for batch_features, batch_labels in train_dataloader:
    
        # 2. Forward pass
        y_pred = model(batch_features)
        # print(y_pred)
        
        # 3. Loss
        loss_res = loss(y_pred, batch_labels.unsqueeze(1))
        
        # 3.1 zero
        optimizer.zero_grad()
        
        # 4. Back pass
        loss_res.backward()
        
        # 5. Params update
        optimizer.step()
        
    print(f"Epoch = {epoch + 1}, loss = {loss_res}")


Epoch = 1, loss = 0.6127213835716248
Epoch = 2, loss = 0.588481605052948
Epoch = 3, loss = 0.543884813785553
Epoch = 4, loss = 0.20025934278964996
Epoch = 5, loss = 0.12304816395044327
Epoch = 6, loss = 0.10048215836286545
Epoch = 7, loss = 0.07001348584890366
Epoch = 8, loss = 0.07067961245775223
Epoch = 9, loss = 0.03420526161789894
Epoch = 10, loss = 0.09616704285144806
Epoch = 11, loss = 0.0993889644742012
Epoch = 12, loss = 0.03192318603396416
Epoch = 13, loss = 0.01651832088828087
Epoch = 14, loss = 0.02305077202618122
Epoch = 15, loss = 0.028876299038529396
Epoch = 16, loss = 0.04870063439011574
Epoch = 17, loss = 0.018399883061647415
Epoch = 18, loss = 0.022588646039366722
Epoch = 19, loss = 0.0016546774422749877
Epoch = 20, loss = 0.02937624230980873
Epoch = 21, loss = 0.07408445328474045
Epoch = 22, loss = 0.1058400347828865
Epoch = 23, loss = 0.046651482582092285
Epoch = 24, loss = 0.06701237708330154
Epoch = 25, loss = 0.04326025769114494
Epoch = 26, loss = 0.03040551953017

Evaluation

In [203]:
model.eval()
accuracy = []


with torch.no_grad():
    
    for batch_features, batch_labels in test_dataloader:
        
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.5).float()
        batch_acc = (y_pred == batch_labels.unsqueeze(1)).float().mean().item()
        accuracy.append(batch_acc)
    
overall = sum(accuracy)/len(accuracy)
print(overall)


0.9895833333333334
